# Agentic RAG Knowledge Assistant
### Built with Amazon Bedrock + Amazon OpenSearch + Amazon S3

**Author:** Personal Demonstration Project  
**Date:** March 2025  
**Purpose:** Demonstrate how agentic and generative AI can improve knowledge discovery and decision-making in regulated environments.

---

## Architecture Overview

This notebook implements a **multi-agent RAG pipeline** with three specialized agents:

| Agent | Responsibility |
|---|---|
| **Planner Agent** | Parses user query, decomposes into sub-tasks, selects retrieval strategy |
| **Retrieval Agent** | Performs semantic/vector search against OpenSearch knowledge base |
| **Evaluation Agent** | Validates response for accuracy, groundedness, and adds citations |

```
User Query
    │
    ▼
┌─────────────────┐
│  Planner Agent  │  ← Amazon Bedrock (Claude / Titan)
└────────┬────────┘
         │ Sub-tasks
         ▼
┌─────────────────┐
│ Retrieval Agent │  ← Amazon OpenSearch (vector search)
└────────┬────────┘         ↑
         │ Evidence    Amazon S3
         ▼           (Knowledge Base)
┌─────────────────┐
│Evaluation Agent │  ← Amazon Bedrock (validation + citations)
└────────┬────────┘
         │
         ▼
  Final Answer + Citations
```

## 1. Install Dependencies

In [ ]:
!pip install boto3 opensearch-py langchain langchain-aws python-dotenv tiktoken --quiet

## 2. Configuration & AWS Setup

In [5]:
import boto3
import json
import os
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

# ── AWS Configuration ──────────────────────────────────────────────────────────
AWS_REGION        = "us-east-1"
S3_BUCKET         = "your-knowledge-base-bucket"
OPENSEARCH_HOST   = "your-opensearch-endpoint"
OPENSEARCH_INDEX  = "knowledge-base-index"
BEDROCK_MODEL_ID  = "anthropic.claude-3-sonnet-20240229-v1:0"
EMBED_MODEL_ID    = "amazon.titan-embed-text-v2:0"

# ── Boto3 Clients ──────────────────────────────────────────────────────────────
session        = boto3.Session(region_name=AWS_REGION)
bedrock_rt     = session.client("bedrock-runtime")
s3_client      = session.client("s3")
credentials    = session.get_credentials().resolved_credentials

awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    AWS_REGION,
    "es",
    session_token=credentials.token
)

# ── OpenSearch Client ──────────────────────────────────────────────────────────
os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection
)

print("✅ AWS clients initialized successfully.")

## 3. Load & Index Documents from S3 into OpenSearch

In [6]:
def get_embedding(text: str) -> list:
    """Generate a vector embedding using Amazon Titan Embeddings via Bedrock."""
    response = bedrock_rt.invoke_model(
        modelId=EMBED_MODEL_ID,
        body=json.dumps({"inputText": text}),
        contentType="application/json",
        accept="application/json"
    )
    body = json.loads(response["body"].read())
    return body["embedding"]


def load_documents_from_s3(bucket: str, prefix: str = "") -> list:
    """Load text documents from S3 bucket."""
    documents = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".txt") or key.endswith(".md"):
                body = s3_client.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8")
                documents.append({"source": key, "content": body})
    print(f"✅ Loaded {len(documents)} documents from S3.")
    return documents


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    """Split text into overlapping chunks for indexing."""
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def index_documents(documents: list):
    """Embed and index document chunks into OpenSearch."""
    # Create index with KNN vector mapping
    index_body = {
        "settings": {"index": {"knn": True}},
        "mappings": {
            "properties": {
                "embedding": {"type": "knn_vector", "dimension": 1024},
                "content":   {"type": "text"},
                "source":    {"type": "keyword"}
            }
        }
    }
    if not os_client.indices.exists(OPENSEARCH_INDEX):
        os_client.indices.create(index=OPENSEARCH_INDEX, body=index_body)
        print(f"✅ Created index: {OPENSEARCH_INDEX}")

    for doc in documents:
        chunks = chunk_text(doc["content"])
        for i, chunk in enumerate(chunks):
            embedding = get_embedding(chunk)
            os_client.index(
                index=OPENSEARCH_INDEX,
                body={"embedding": embedding, "content": chunk, "source": doc["source"]},
                id=f"{doc['source']}_{i}"
            )
    print(f"✅ Indexed {len(documents)} documents into OpenSearch.")


# ── Run Indexing Pipeline ──────────────────────────────────────────────────────
# documents = load_documents_from_s3(S3_BUCKET)
# index_documents(documents)
print("ℹ️  Indexing pipeline ready. Uncomment above lines to run.")

ℹ️  Indexing pipeline ready. Uncomment above lines to run.


## 4. Agent 1 — Planner Agent
Decomposes the user query into focused sub-questions for targeted retrieval.

In [7]:
def planner_agent(user_query: str) -> list:
    """
    Planner Agent: Uses Amazon Bedrock to decompose a complex user query
    into focused sub-questions for targeted retrieval.
    """
    system_prompt = """You are a Planner Agent in an Agentic RAG system.
Your job is to analyze a user query and decompose it into 2-4 focused sub-questions
that together will fully answer the original question.
Respond ONLY with a JSON array of sub-questions. Example:
["What is X?", "How does Y work?", "What are the requirements for Z?"]"""

    payload = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "system": system_prompt,
        "messages": [{"role": "user", "content": user_query}]
    }

    response = bedrock_rt.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=json.dumps(payload),
        contentType="application/json",
        accept="application/json"
    )
    result = json.loads(response["body"].read())
    sub_questions = json.loads(result["content"][0]["text"])

    print(f"📋 Planner Agent decomposed query into {len(sub_questions)} sub-questions:")
    for i, q in enumerate(sub_questions, 1):
        print(f"   {i}. {q}")
    return sub_questions


# ── Demo ───────────────────────────────────────────────────────────────────────
sample_query = "What are the compliance requirements for data retention in financial services and how should incidents be reported?"
# sub_questions = planner_agent(sample_query)
print("ℹ️  Planner Agent ready. Uncomment above line to run with live Bedrock.")

ℹ️  Planner Agent ready. Uncomment above line to run with live Bedrock.


## 5. Agent 2 — Retrieval Agent
Performs semantic vector search against the OpenSearch knowledge base.

In [4]:
def retrieval_agent(sub_questions: list, top_k: int = 3) -> list:
    """
    Retrieval Agent: Embeds each sub-question and performs KNN vector search
    against the OpenSearch index to retrieve relevant document chunks.
    Supports multi-hop inference by querying for each sub-question.
    """
    all_evidence = []
    seen_ids     = set()

    for question in sub_questions:
        query_embedding = get_embedding(question)

        search_body = {
            "size": top_k,
            "query": {
                "knn": {
                    "embedding": {
                        "vector": query_embedding,
                        "k": top_k
                    }
                }
            },
            "_source": ["content", "source"]
        }

        results = os_client.search(index=OPENSEARCH_INDEX, body=search_body)

        for hit in results["hits"]["hits"]:
            doc_id = hit["_id"]
            if doc_id not in seen_ids:   # Deduplicate
                seen_ids.add(doc_id)
                all_evidence.append({
                    "id":      doc_id,
                    "source":  hit["_source"]["source"],
                    "content": hit["_source"]["content"],
                    "score":   hit["_score"]
                })

    # Sort by relevance score
    all_evidence.sort(key=lambda x: x["score"], reverse=True)

    print(f"🔍 Retrieval Agent found {len(all_evidence)} unique evidence chunks.")
    for e in all_evidence[:3]:
        print(f"   Source: {e['source']} | Score: {e['score']:.4f}")
        print(f"   Preview: {e['content'][:120]}...\n")

    return all_evidence


print("ℹ️  Retrieval Agent ready.")

ℹ️  Retrieval Agent ready.


## 6. Agent 3 — Evaluation Agent
Validates the response for groundedness and generates a cited, explainable answer.

In [3]:
def evaluation_agent(user_query: str, evidence: list) -> dict:
    """
    Evaluation Agent: Uses Amazon Bedrock to synthesize retrieved evidence into
    a grounded, cited, and validated response. Reduces hallucination by
    constraining generation strictly to retrieved context.
    """
    # Format evidence for the prompt
    evidence_text = ""
    for i, chunk in enumerate(evidence, 1):
        evidence_text += f"[Source {i}: {chunk['source']}]\n{chunk['content']}\n\n"

    system_prompt = """You are an Evaluation Agent in an Agentic RAG system for a regulated environment.
Your responsibilities:
1. Answer the user's question using ONLY the provided evidence chunks.
2. Cite your sources using [Source N] notation inline.
3. If the evidence does not fully support an answer, clearly state what is and is not supported.
4. Do NOT hallucinate or introduce information not present in the evidence.
5. End your response with a 'Confidence Assessment' (High / Medium / Low) and brief justification."""

    user_prompt = f"""User Question: {user_query}

Evidence:
{evidence_text}

Please provide a grounded, cited answer."""

    payload = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1024,
        "system": system_prompt,
        "messages": [{"role": "user", "content": user_prompt}]
    }

    response = bedrock_rt.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=json.dumps(payload),
        contentType="application/json",
        accept="application/json"
    )
    result       = json.loads(response["body"].read())
    final_answer = result["content"][0]["text"]

    return {
        "query":    user_query,
        "answer":   final_answer,
        "sources":  [e["source"] for e in evidence],
        "num_sources_used": len(evidence)
    }


print("ℹ️  Evaluation Agent ready.")

ℹ️  Evaluation Agent ready.


## 7. Full Agentic RAG Pipeline
Orchestrates all three agents end-to-end.

In [2]:
def agentic_rag_pipeline(user_query: str) -> dict:
    """
    Full Agentic RAG Pipeline:
    1. Planner Agent  → decomposes query into sub-questions
    2. Retrieval Agent → multi-hop semantic search across knowledge base
    3. Evaluation Agent → validates, grounds, and cites the final answer
    """
    print("=" * 60)
    print(f"🚀 Starting Agentic RAG Pipeline")
    print(f"📝 Query: {user_query}")
    print("=" * 60)

    # Step 1: Plan
    print("\n── Step 1: Planner Agent ──")
    sub_questions = planner_agent(user_query)

    # Step 2: Retrieve
    print("\n── Step 2: Retrieval Agent ──")
    evidence = retrieval_agent(sub_questions, top_k=3)

    # Step 3: Evaluate & Generate
    print("\n── Step 3: Evaluation Agent ──")
    result = evaluation_agent(user_query, evidence)

    print("\n" + "=" * 60)
    print("✅ FINAL ANSWER")
    print("=" * 60)
    print(result["answer"])
    print(f"\n📚 Sources used: {result['num_sources_used']}")
    for src in result["sources"]:
        print(f"   - {src}")

    return result


# ── Run the Pipeline ───────────────────────────────────────────────────────────
query = "What are the data retention and incident reporting requirements for regulated financial services?"
# result = agentic_rag_pipeline(query)
print("ℹ️  Full pipeline ready. Uncomment above line to run end-to-end.")

ℹ️  Full pipeline ready. Uncomment above line to run end-to-end.


## 8. Mock Demo (No AWS Credentials Required)
Simulates the full pipeline output for demonstration purposes.

In [1]:
def mock_demo():
    """Simulates the Agentic RAG pipeline without live AWS credentials."""

    query = "What are the data retention and incident reporting requirements for regulated financial services?"

    print("=" * 60)
    print("🚀 Agentic RAG Knowledge Assistant — Mock Demo")
    print(f"📝 Query: {query}")
    print("=" * 60)

    # Simulated Planner Output
    sub_questions = [
        "What are the data retention requirements in financial services?",
        "What are the incident reporting obligations for financial institutions?",
        "What regulatory bodies govern data retention and incident reporting?"
    ]
    print("\n── Step 1: Planner Agent ──")
    print(f"📋 Decomposed into {len(sub_questions)} sub-questions:")
    for i, q in enumerate(sub_questions, 1):
        print(f"   {i}. {q}")

    # Simulated Retrieval Output
    evidence = [
        {"source": "s3://kb/compliance_policy_v3.txt",   "score": 0.9823, "content": "Financial institutions must retain transaction records for a minimum of 7 years per SEC Rule 17a-4..."},
        {"source": "s3://kb/incident_response_sop.txt",  "score": 0.9541, "content": "Material cybersecurity incidents must be reported to regulators within 72 hours of discovery per FDIC guidelines..."},
        {"source": "s3://kb/gdpr_financial_guidance.txt", "score": 0.9102, "content": "Under GDPR Article 30, firms must maintain records of processing activities including retention schedules..."},
    ]
    print("\n── Step 2: Retrieval Agent ──")
    print(f"🔍 Retrieved {len(evidence)} evidence chunks:")
    for e in evidence:
        print(f"   Source: {e['source']} | Score: {e['score']}")
        print(f"   Preview: {e['content'][:100]}...\n")

    # Simulated Evaluation Output
    final_answer = """
Based on the retrieved knowledge base documents:

**Data Retention Requirements:**
Financial institutions are required to retain transaction records for a minimum of **7 years**
in accordance with SEC Rule 17a-4 [Source 1]. Additionally, under GDPR Article 30, firms must
maintain detailed records of all processing activities, including defined retention schedules [Source 3].

**Incident Reporting Obligations:**
Material cybersecurity incidents must be reported to the relevant regulatory body within **72 hours**
of discovery, as mandated by FDIC guidelines [Source 2].

**Confidence Assessment: High**
All claims are directly supported by retrieved source documents with high relevance scores (>0.90).
    """
    print("── Step 3: Evaluation Agent ──")
    print("\n" + "=" * 60)
    print("✅ FINAL ANSWER (Grounded + Cited)")
    print("=" * 60)
    print(final_answer)
    print(f"📚 Sources used: {len(evidence)}")
    for e in evidence:
        print(f"   - {e['source']}")


mock_demo()

🚀 Agentic RAG Knowledge Assistant — Mock Demo
📝 Query: What are the data retention and incident reporting requirements for regulated financial services?

── Step 1: Planner Agent ──
📋 Decomposed into 3 sub-questions:
   1. What are the data retention requirements in financial services?
   2. What are the incident reporting obligations for financial institutions?
   3. What regulatory bodies govern data retention and incident reporting?

── Step 2: Retrieval Agent ──
🔍 Retrieved 3 evidence chunks:
   Source: s3://kb/compliance_policy_v3.txt | Score: 0.9823
   Preview: Financial institutions must retain transaction records for a minimum of 7 years per SEC Rule 17a-4.....

   Source: s3://kb/incident_response_sop.txt | Score: 0.9541
   Preview: Material cybersecurity incidents must be reported to regulators within 72 hours of discovery per FDI...

   Source: s3://kb/gdpr_financial_guidance.txt | Score: 0.9102
   Preview: Under GDPR Article 30, firms must maintain records of processing act

## 9. Summary — Key Technical Decisions

| Component | Technology | Rationale |
|---|---|---|
| Foundation Model | Amazon Bedrock (Claude 3 Sonnet) | Enterprise-grade LLM with strong instruction-following |
| Embeddings | Amazon Titan Embed v2 | Natively integrated with Bedrock, 1024-dim vectors |
| Vector Store | Amazon OpenSearch (KNN) | Scalable, managed, AWS-native semantic search |
| Knowledge Base | Amazon S3 | Durable, scalable document storage |
| Agentic Pattern | Planner → Retrieval → Evaluation | Separates concerns; enables multi-hop reasoning |
| Hallucination Mitigation | Grounded prompting + citation enforcement | Ensures all claims are traceable to source documents |

---
